# PKG reference data

**PNC Treasury Management · DSI · Payment Knowledge Graph**

Builds the lookup tables that the counterparty and customer metrics join to. Run it after `pkg_edges_refresh.ipynb` and
before the metrics notebooks. Every run rebuilds every table from all months of the edge tables.

| Table | Grain | Was built by |
|---|---|---|
| `pkg_cpty_dim` | one row per counterparty id | `pkg_edges_2024_backfill.ipynb` §5 (same logic) |
| `pkg_cpty_entity_map_v2` | one row per counterparty id: `ent_l0`, `ent_l1`, `ent_l2`, `ent_l2_fi`, rule, name class | `pkg_cpty_entity_map_v2.ipynb` |
| `pkg_fi_routing` | one row per 9-digit routing number: canonical bank name | `pkg_fi_canonical.ipynb` |
| `pkg_fi_name_map` | raw bank name → canonical bank name | `pkg_fi_canonical.ipynb` |
| `pkg_cust_dim` | one row per `mdm_id` | `pkg_customer_metrics_v3.ipynb` (same logic) |

**Changes from the earlier notebooks**
- `ent_l1` (6C P2P ids folded into their 5C id) is built here, so the v1 map `pkg_cpty_entity_map` is no longer needed
  and no longer updated.
- **One row per counterparty id in the map:** if an id ever has more than one row, the lowest `ent_l2` is kept and §3
  reports it.
- **Name keys ignore spaces** (`KEY_IGNORE_SPACES`): HARLEYDAVIDSON and HARLEY DAVIDSON are one merge key, so the brand
  merges across banks when any of its spellings is distinctive.
- **Bank-name rules** (`FI_FIX_RULES`): every raw bank name containing VETERANS AFFAIRS takes the name of the US
  Treasury found in the data, so federal payments (VACP TREAS 310) no longer flip between two banks. §4 shows which names
  were matched.
- The review sections (typer eyeball tables, v1 → v2 crosstab, bank impact and bank-swap history) are left out; each
  section ends with a short QA.

**How it runs.** Each table is written to a staging folder; §6 publishes all of them together at the end, so a run that
stops halfway leaves the previous tables in place. Rerun all cells after a failure.

**Downstream.** The counterparty and customer metrics notebooks treat new versions of these tables as a reason to
rebuild from 2024-01.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB         = "bdahd01p_dlcdi1_cdi_tm"
CUST_TABLE = "dsihd01p_dsi.neo4j_customer"
SHUFFLE_PARTITIONS = 2000

# ---- Counterparty dimension (pkg_cpty_dim): flags and name key, unchanged
NAME_KEY_MINLEN = 4
DIM_BIZ = (r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|BANK|TRUST|FUND|"
           r"FOUNDATION|ASSOC|ASSOCIATION|SERVICES?|GROUP|HOLDINGS|PARTNERS|UNIVERSITY|COLLEGE|SCHOOL|DISTRICT|CITY|COUNTY|"
           r"STATE|TOWNSHIP|BOROUGH|HOSPITAL|CENTER|CHURCH|DEPT|DEPARTMENT|TREASURER|INSURANCE|SUPPLY|ENTERPRISES|"
           r"INTERNATIONAL|AUTHORITY|SYSTEMS|SOLUTIONS|TECHNOLOGIES|MANAGEMENT|PROPERTIES|REALTY|CONSTRUCTION)\b")
DIM_LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"
DIM_FI_LIKE = r"\b(BANK|BANCORP|BANCSHARES|N A|NATIONAL ASSOCIATION|CREDIT UNION|FCU|SAVINGS|TRUST COMPANY)\b"
DIM_PLACEHOLDER_KEYS = ["BANK ACCOUNT NAME", "ACCOUNT NAME", "UNKNOWN", "UNKNOWN PAYEE", "NONE", "NULL", "N A", "NA",
                        "TEST", "PAYEE", "VENDOR", "CUSTOMER", "ACCOUNT", "CASH", "CHECK", "DEPOSIT", "TRANSFER",
                        "PAYMENT", "REFUND", "MISC", "MISCELLANEOUS", "VARIOUS", "NAME"]

# ---- ent_l1: a 6C id joins its 5C id when at least this share of its dedup pairs point to that 5C id
XW_MIN_PURITY = 0.8

# ---- Typer v2 and entity map v2 (unchanged unless noted)
FN_MIN_IDS     = 200        # a first name must lead at least this many P2P names …
FN_LEAD_RATIO  = 3.0        # … and lead ≥ this many times more often than it trails
KEY_MIN_LEN    = 4          # shortest cleaned business key allowed to merge
WIRE_INST_MIN  = 1_000_000  # wire-only ids with at least this many lifetime dollars are institutions
NUM_SHARE      = 0.8        # digits / alphanumerics at or above this → numeric_name
PERSON_MAX_AMT = 10_000_000 # person rules apply only below this lifetime amount
VOCAB_MIN_IDS  = 300        # learned business word: in ≥ this many legal-suffix / org-word names …
VOCAB_RATIO    = 20.0       # … and ≥ this many times more often than in P2P person names
DF_MAX         = 300        # a key is distinctive if it has a word (≥ 4 letters) in ≤ this many distinct keys
CONC_MIN       = 20         # … or if it averages ≥ this many ids per FI
KEY_IGNORE_SPACES = True    # NEW: spaces ignored in merge keys (HARLEYDAVIDSON = HARLEY DAVIDSON)
NO_MERGE_KEYS  = []
SHORT_KEYS_OK  = ["UPS", "BP", "DHL", "HP", "IBM", "GE", "3M", "CVS", "ATT", "UBS", "AIG", "HSN", "QVC", "KFC", "UMR"]
BRAND_BEH_SHARE = 0.8
KEY_ALIASES = {r"^(AMAZON|AMZN)\b.*": "AMAZON", r"^(MICROSOFT|MSFT)\b.*": "MICROSOFT",
               r"^(SOUTHWES\w*|SWA)\b.*": "SOUTHWEST AIRLINES", r"^(FACEBK|FACEBOOK)\b.*": "FACEBOOK"}
KEEP_NUM_AFTER = ["POST", "LOCAL", "CHAPTER", "LODGE", "TROOP", "COUNCIL", "DISTRICT", "WARD"]
TEST_NAME_RE   = r"SMOKE ?TEST|\bTEST(ER)? ?(USER|ACCOUNT|ACCT|NAME|\d+)\b"
LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LLLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY",
         "PC", "PLLC", "PA", "PLC", "SCSP", "SCS", "SARL", "SA", "AG", "GMBH", "BV", "NV", "SPA", "SRL", "PTE", "PTY",
         "ULC", "DBA", "LTDA", "SAS", "KG", "AB", "OYJ", "NA"]
ORG = ["HOLDINGS", "HOLDING", "FUND", "FUNDS", "CAPITAL", "PARTNERS", "PARTNERSHIP", "INVESTORS", "INVESTMENT",
       "INVESTMENTS", "MANAGEMENT", "AGGREGATOR", "ACCOUNT", "COLLECTION", "COLLECTIONS", "ESCROW", "TITLE",
       "SETTLEMENT", "TRUST", "TRUSTEE", "ESTATE OF", "CLINIC", "HOSPITAL", "HEALTH", "HEALTHCARE", "MEDICAL",
       "DENTAL", "PHARMACY", "UNIVERSITY", "COLLEGE", "SCHOOL", "SCHOOLS", "ACADEMY", "CHURCH", "MINISTRIES",
       "MINISTRY", "PARISH", "DIOCESE", "TEMPLE", "MOSQUE", "ASSOC", "ASSOCIATION", "ASSN", "ASSOCIATES", "FOUNDATION",
       "SOCIETY", "INSTITUTE", "CENTER", "CENTRE", "CLUB", "LEAGUE", "COUNCIL", "COMMITTEE", "UNION", "SERVICES",
       "SERVICE", "SOLUTIONS", "GROUP", "ENTERPRISES", "ENTERPRISE", "INDUSTRIES", "INTERNATIONAL", "SYSTEMS",
       "TECHNOLOGIES", "TECHNOLOGY", "CONSTRUCTION", "CONTRACTING", "CONTRACTORS", "BUILDERS", "REALTY",
       "PROPERTIES", "PROPERTY", "APARTMENTS", "HOMEOWNERS", "HOA", "CONDO", "CONDOMINIUM", "INSURANCE", "AGENCY",
       "MORTGAGE", "LENDING", "LOAN", "LOANS", "FINANCIAL", "FINANCE", "PAYROLL", "PAYMENTS", "TREASURY",
       "DEPARTMENT", "DEPT", "COUNTY", "CITY OF", "STATE OF", "TOWN OF", "VILLAGE OF", "TOWNSHIP", "BOROUGH",
       "AUTHORITY", "DISTRICT", "COMMISSION", "COMMONWEALTH", "GOVERNMENT", "UNITED STATES", "US DEPT", "IRS",
       "UTILITIES", "UTILITY", "ELECTRIC", "ENERGY", "POWER", "WATER", "GAS", "TELECOM", "WIRELESS",
       "COMMUNICATIONS", "MOTORS", "AUTOMOTIVE", "AUTO", "RESTAURANT", "CAFE", "GRILL", "PIZZA", "MARKET", "STORE",
       "STORES", "SHOP", "SUPPLY", "SUPPLIES", "DISTRIBUTION", "DISTRIBUTORS", "WHOLESALE", "LOGISTICS",
       "TRANSPORT", "TRANSPORTATION", "TRUCKING", "FREIGHT", "EXPRESS", "FARM", "FARMS", "LAW", "LEGAL", "ATTORNEYS",
       "CPA", "CPAS", "ACCOUNTING", "CONSULTING", "CONSULTANTS", "DESIGN", "STUDIO", "STUDIOS", "MEDIA", "MARKETING",
       "PRODUCTIONS", "VENTURES", "EQUITY", "ADVISORS", "ADVISORY", "SECURITIES", "BROKERAGE", "REIT", "LEASING",
       "RENTAL", "RENTALS", "PLUMBING", "ELECTRICAL", "ROOFING", "HVAC", "LANDSCAPING", "CLEANING", "CORPORATE",
       "TOPCO", "BIDCO", "HOLDCO", "OPCO", "ACQUISITION", "MERGER", "SUB", "SPV", "NOMINEE", "OPERATING", "PRODUCTS",
       "MANUFACTURING", "PROCESSING", "PROCESSOR", "PROCESSORS", "LABS", "LABORATORIES", "PHARMACEUTICALS", "SOFTWARE",
       "NETWORKS", "AIRLINES", "HOTEL", "HOTELS", "RESORT", "CASINO", "THEATRE", "THEATER", "MUSEUM", "LIBRARY",
       "PROGRAM", "PROJECT", "PLAN", "FEDERAL", "NATIONAL", "AMERICA", "AMERICAN", "UNITED", "GENERAL", "CREDIT UNION",
       "REVENUE", "TREASURER", "CUSTOMERS", "GATEWAY", "GATEWAYS", "MERCHANT", "RECEIPT", "WIRES", "TRANSACTION",
       "TRANSACTIONS", "CASH CARRY", "AIRCRAFT", "ENGINES", "AIR LINES", "INTERACTIVE", "DISTRIBUTORS"]
ORG_BRANDS = ["PAYMENTECH", "PAYPAL", "WORLDPAY", "FISERV"]
CHANNEL = ["OTP", "IVR", "WEB", "PPD", "CCD", "WEBPAY", "AUTOPAY", "EPAY", "ONLINE", "PMT", "PMTS", "PYMT", "PAYMENT",
           "PAYMENTS", "BILLPAY", "BILL PAY", "ACH", "DEBIT", "RECURRING"]
STOP = ["AND", "OF", "THE", "FOR", "DE", "LA", "DEL", "LOS", "LAS", "Y", "DA", "DOS", "VAN", "DER", "MC", "ST", "LE",
        "DI", "AL", "EL", "BIN", "BEN", "ABU"]
PERSON_EXTRA = ["MR", "MRS", "MS", "MISS", "DR", "JR", "SR", "II", "III", "IV", "MD", "DO", "DDS", "DMD", "DVM", "DC",
                "OD", "RN", "NP", "PHD", "ESQ"]
NOT_FIRST = ["THE", "AND", "OF", "MY", "YOUR", "OUR", "NEW", "FIRST", "BEST", "SUPER", "GREAT", "AMERICAN", "NATIONAL",
             "US", "USA", "ALL", "ONE", "BIG", "LITTLE", "GOOD", "TOP", "PRO", "ROYAL", "GOLDEN", "BLUE", "RED",
             "GREEN", "BLACK", "WHITE", "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "SAINT", "ST", "MOUNT", "HOLY",
             "GRACE", "LOVE", "PAYPAL", "VENMO", "ZELLE", "CASH"]
GENERIC_KEYS = ["PAYMENT", "PAYMENTS", "TRANSFER", "DEPOSIT", "REFUND", "CHECK", "WIRE", "ACH", "ONLINE", "MOBILE",
                "BILL", "BILLPAY", "PAYEE", "VENDOR", "CUSTOMER", "CLIENT", "OWNER", "TENANT", "LANDLORD", "USER",
                "CHECKING", "SAVINGS", "ACCT", "ACCOUNT", "MONEY MARKET"]
PLACEHOLDERS = ["BANK ACCOUNT NAME", "UNKNOWN", "UNKNOWN NAME", "NONE", "NA", "N A", "NULL", "TEST", "CUSTOMER",
                "BENEFICIARY", "PAYEE", "ACCOUNT HOLDER", "NAME", "NO NAME", "ON FILE", "CASH", "VARIOUS", "XXX",
                "USER", "CHECKING", "SAVINGS", "ACCT", "CHECKING ACCOUNT", "SAVINGS ACCOUNT", "MY ACCOUNT"]
STREET = ["ROAD", "RD", "STREET", "ST", "AVENUE", "AVE", "AV", "BOULEVARD", "BLVD", "DRIVE", "DR", "LANE", "LN",
          "HIGHWAY", "HWY", "WAY", "COURT", "CT", "PLACE", "PL", "PARKWAY", "PKWY", "CIRCLE", "CIR", "TERRACE", "TER",
          "TRAIL", "TRL", "PIKE", "ROUTE", "RT", "RTE", "SQUARE", "SQ", "SUITE", "STE", "APT", "UNIT", "FLOOR", "FL"]

# ---- Canonical bank names (unchanged unless noted)
FI_FIXES = {"ALLLY BANK": "ALLY BANK"}          # raw FI name → corrected name (exact match)
# NEW: (pattern for raw names to fix, pattern for the target name). Every raw FI name matching the first pattern takes
# the raw FI name matching the second pattern that has the most dollars. Banks and credit unions never match. §4 shows
# what was matched.
VA_NAME_RE       = r"^(?!.*\b(CREDIT UNION|FCU|BANK)\b).*\bVETERANS AFFAIRS\b"
TREASURY_NAME_RE = r"^(?!.*\b(CREDIT UNION|FCU|BANK)\b).*\b(DEPARTMENT OF THE TREASURY|DEPT OF THE TREASURY|US TREASURY)\b"
FI_FIX_RULES = [(VA_NAME_RE, TREASURY_NAME_RE)]
FI_PARTNER_RE = (r"^.+?\s+(?:WITH|PARTNERING WITH|POWERED BY|IN PARTNERSHIP WITH|THROUGH)\s+"
                 r"(.*\b(?:BANK|CREDIT UNION|TRUST|N A|NA|FSB|SSB|SAVINGS|BANCORP|FCU)\b.*)$")
FI_DIVISION_RE = r"^.+?\s+(?:ADO|A DIVISION OF|DIVISION OF|A DIV OF|DIV OF)\s+(.{4,})$"
FI_KEEP_RAW_ROUTINGS = []
NAME_MIN_SHARE = 0.9
FI_STRIP = ["THE", "NA", "NATIONAL ASSOCIATION", "INC", "CORP", "CORPORATION", "CO", "COMPANY", "FSB", "SSB",
            "FEDERAL CREDIT UNION", "CREDIT UNION", "FCU", "LLC"]

# ---- Customer dimension (unchanged)
NAICS_PLACEHOLDERS = ["******", "-1", "0", "00", "000000", "999999", "UNKNOWN", "N/A", "NA", "NULL", "NONE"]

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import re, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_reference_data")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write.
# It doesn't affect the data; its loggers are switched off so the cell outputs stay readable.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def mkdirs(p):
    fs_for(p).mkdirs(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    assert not exists(b), f"rename target already exists: {b}"
    if not fs_for(a).rename(JPath(a), JPath(b)):
        raise RuntimeError(f"rename failed: {a} -> {b}")
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def nz(c):
    """String, trimmed, blank -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

## §0 Inputs

The edge tables must exist, and every month of `pkg_cpty_month` must have been published by `pkg_edges_refresh.ipynb`
(build marker present). The customer table must have the columns the customer dimension uses.

In [ ]:
# ---- §0 Inputs -------------------------------------------------------------------------------------
T_EDGE_M, T_CPTY_MONTH, T_DEDUP = f"{DB}.pkg_edge_monthly", f"{DB}.pkg_cpty_month", f"{DB}.pkg_dedup_5c6c"
OUT_KEYS = ["cpty_dim", "cpty_entity_map_v2", "fi_routing", "fi_name_map", "cust_dim"]
T = {k: f"{DB}.pkg_{k}" for k in OUT_KEYS}
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
P = {k: f"{DB_LOC}/pkg_{k}" for k in OUT_KEYS}
STAGE = f"{DB_LOC}/_staging_pkg_reference"
TMP   = f"{DB_LOC}/_tmp_pkg_reference"
S = {k: f"{STAGE}/{k}" for k in OUT_KEYS}

def table_exists(t):
    db, name = t.split(".")
    return len(spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").collect()) > 0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")

PROBLEMS = []
NEED = {T_EDGE_M: ["src_id", "src_kind", "dst_id", "dst_kind", "month"],
        T_CPTY_MONTH: ["cpty_id", "cpty_id_src", "category", "cpty_type", "cpty_name", "fi", "n_txn", "amount", "month"],
        T_DEDUP: ["cpty_id_drop", "cpty_id_keep", "month"],
        CUST_TABLE: ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "customer_start_dt",
                     "naics_cd", "naics_desc", "load_dt"]}
for t, cols in NEED.items():
    if not table_exists(t):
        PROBLEMS.append({"problem": "table missing", "detail": t}); continue
    have = {c.lower() for c in spark.table(t).columns}
    for c in cols:
        if c not in have:
            PROBLEMS.append({"problem": "column missing", "detail": f"{t}.{c}"})

MONTHS = []
if table_exists(T_CPTY_MONTH):
    MONTHS = sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {T_CPTY_MONTH}").collect())
    _cm_loc = table_location(T_CPTY_MONTH)
    _unpub = [m for m in MONTHS if not exists(f"{_cm_loc}/month={m}/_PKG_BUILD")]
    if _unpub:
        PROBLEMS.append({"problem": "months not published by pkg_edges_refresh",
                         "detail": ", ".join(_unpub)})
if PROBLEMS:
    out(sec(f"§0 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§0: {len(PROBLEMS)} problem(s), see the table above")

delete(STAGE); delete(TMP)                       # leftovers of an earlier run
_ex = pd.DataFrame([{"table": T[k], "exists": table_exists(T[k]),
                     "location": table_location(T[k]) if table_exists(T[k]) else P[k]} for k in OUT_KEYS])
out(sec(f"Inputs: edge tables {MONTHS[0]} → {MONTHS[-1]} ({len(MONTHS)} months, all published)",
        tbl(pd.DataFrame([{"input": t} for t in NEED]))),
    sec("Tables this notebook replaces in §6", tbl(_ex)))
_S0_OK = True

## §1 Counterparty dimension: `pkg_cpty_dim`

One row per counterparty id over every month of `pkg_cpty_month`: dominant name, bank and type (by transactions), activity
span, categories, payee / payer flags, and the v1 name flags (`is_business`, `name_is_fi`, `is_placeholder`) that the
typer in §2 reads. Same logic as the old edge notebook.

In [ ]:
# ---- §1 Counterparty dimension ---------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

clean = lambda c: F.trim(F.regexp_replace(F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " "), r"\s+", " "))
def raw_key(c):
    return F.trim(F.regexp_replace(F.regexp_replace(clean(c), DIM_LEGAL, " "), r"\s+", " "))

with timed("§1 counterparty dimension"):
    cm_all = spark.table(T_CPTY_MONTH)
    def dominant(col):
        w = Window.partitionBy("cpty_id").orderBy(F.desc("n"), F.col(col))
        return (cm_all.filter(F.col(col).isNotNull()).groupBy("cpty_id", col).agg(F.sum("n_txn").alias("n"))
                      .withColumn("r", F.row_number().over(w)).filter("r = 1").select("cpty_id", col))
    nvar = (cm_all.filter(F.col("cpty_name").isNotNull()).groupBy("cpty_id")
                  .agg(F.countDistinct("cpty_name").alias("n_name_variants")))
    letter = F.regexp_extract(F.split("category", r"\.").getItem(0), "([ABC])$", 1)
    base = cm_all.groupBy("cpty_id").agg(
               F.max("cpty_id_src").alias("cpty_id_src"), F.sum("n_txn").alias("n_txn"),
               F.sum("amount").cast("decimal(22,2)").alias("amount"),
               F.min("month").alias("first_month"), F.max("month").alias("last_month"),
               F.countDistinct("month").alias("active_months"),
               F.concat_ws(",", F.sort_array(F.collect_set(F.split("category", r"\.").getItem(0)))).alias("categories"),
               F.max((letter == "B").cast("int")).cast("boolean").alias("is_payee"),
               F.max((letter == "C").cast("int")).cast("boolean").alias("is_payer"))
    rk = raw_key(F.col("cpty_name"))
    dim = (base.join(dominant("cpty_name"), "cpty_id", "left").join(dominant("fi"), "cpty_id", "left")
               .join(dominant("cpty_type"), "cpty_id", "left").join(nvar, "cpty_id", "left")
               .withColumn("is_business",    F.coalesce(clean(F.col("cpty_name")).rlike(DIM_BIZ), F.lit(False)))
               .withColumn("name_is_fi",     F.coalesce(clean(F.col("cpty_name")).rlike(DIM_FI_LIKE), F.lit(False)))
               .withColumn("is_placeholder", F.coalesce(rk.isin(DIM_PLACEHOLDER_KEYS)
                                                        | clean(F.col("cpty_name")).isin(DIM_PLACEHOLDER_KEYS), F.lit(False)))
               .withColumn("name_key", F.when(~F.col("is_placeholder") & (F.length(rk) >= NAME_KEY_MINLEN), rk)))
    dim.write.mode("overwrite").parquet(S["cpty_dim"])
    DIM = spark.read.parquet(S["cpty_dim"])

q = DIM.agg(F.count("*").alias("ids"), F.countDistinct("cpty_id").alias("distinct_ids"),
            F.sum("amount").cast("double").alias("amount"),
            F.sum(F.col("is_business").cast("long")).alias("business_like"),
            F.sum(F.col("name_is_fi").cast("long")).alias("bank_like_names"),
            F.sum(F.col("is_placeholder").cast("long")).alias("placeholder_names"),
            F.min("first_month").alias("first_month"), F.max("last_month").alias("last_month")).toPandas()
assert int(q.ids[0]) == int(q.distinct_ids[0]), "pkg_cpty_dim must have one row per cpty_id"
out(sec(f"§1 pkg_cpty_dim (staged, {TIMES['§1 counterparty dimension']:.1f} min)", tbl(q)))

## §2 `ent_l1` and name typer v2

- **`ent_l1`:** a 6C P2P id is folded into the 5C id it was paired with in the 5C/6C dedup, when at least
  `XW_MIN_PURITY` of its pairs point to that id. Every other id is its own `ent_l1`.
- **Typer v2:** every id gets a `name_class` (business / person / bank / numeric_name / placeholder / no_name /
  unknown) and the rule that set it. First names and learned business words come from the data. Same rules as
  `pkg_cpty_entity_map_v2.ipynb`.

In [ ]:
# ---- §2 ent_l1 and typer v2 ------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def wre(words):                       # whole-word alternation
    return r"\b(" + "|".join(sorted(set(words), key=len, reverse=True)) + r")\b"

with timed("§2 ent_l1 + typer"):
    # ent_l1: 6C → 5C crosswalk from the dedup pairs
    xw = (spark.table(T_DEDUP).filter(F.col("cpty_id_drop").isNotNull() & F.col("cpty_id_keep").isNotNull())
               .groupBy("cpty_id_drop", "cpty_id_keep").agg(F.count("*").alias("pairs")))
    wx = Window.partitionBy("cpty_id_drop").orderBy(F.desc("pairs"), "cpty_id_keep")
    xw = (xw.withColumn("tot", F.sum("pairs").over(Window.partitionBy("cpty_id_drop")))
            .withColumn("r", F.row_number().over(wx)).filter("r = 1")
            .filter(F.col("pairs") / F.col("tot") >= XW_MIN_PURITY)
            .select(F.col("cpty_id_drop").alias("cpty_id"), F.col("cpty_id_keep").alias("xw_to")))
    l1 = (DIM.select("cpty_id", "cpty_id_src").join(xw, "cpty_id", "left")
             .withColumn("ent_l1", F.coalesce("xw_to", "cpty_id")).select("cpty_id", "cpty_id_src", "ent_l1"))

    base = (DIM.select("cpty_id", F.col("cpty_name").alias("name"), "fi", "cpty_type", F.col("categories").alias("cats"),
                       F.col("amount").cast("double").alias("amt"), F.col("n_txn").cast("long").alias("n_txn"),
                       F.col("name_is_fi").cast("boolean").alias("v1_fi"),
                       F.col("is_placeholder").cast("boolean").alias("v1_ph"))
               .join(l1, "cpty_id", "inner"))

    # normalised names
    up   = F.upper(F.trim(F.col("name")))
    nm0  = F.trim(F.regexp_replace(F.regexp_replace(up, r"[^A-Z0-9]+", " "), r"\s+", " "))
    ADDR = (r"^(.*?[A-Z].*?)\s+(?:\d{1,6}[A-Z]?\s+(?:[A-Z0-9]+\s+){0,5}?(?:" + "|".join(STREET)
            + r")\b|P\s?O\s+BOX\b|BOX\s+\d).*$")
    nm   = F.trim(F.regexp_replace(nm0, ADDR, "$1"))
    nm_c = F.regexp_replace(nm, r"\b([A-Z]) (?=[A-Z]\b)", "$1")
    nm_p = F.trim(F.regexp_replace(F.regexp_replace(nm, wre(PERSON_EXTRA), " "), r"\s+", " "))
    after_comma = F.trim(F.regexp_replace(F.regexp_replace(F.upper(F.substring_index(F.col("name"), ",", -1)),
                                                           r"[^A-Z]+", " "), r"\s+", " "))
    named = (base
             .withColumn("nm", nm).withColumn("nm_c", nm_c).withColumn("nm_p", nm_p)
             .withColumn("has_comma", F.col("name").contains(","))
             .withColumn("ft1", F.element_at(F.split(F.col("nm_p"), " "), 1))
             .withColumn("ftc", F.when(F.col("has_comma"), F.element_at(F.split(after_comma, " "), 1)))
             .withColumn("n_tok", F.size(F.split(F.col("nm_p"), " ")))
             .withColumn("alpha", F.col("nm_p").rlike(r"^[A-Z]+( [A-Z]+)*$"))
             .withColumn("biz_tok", F.col("nm_c").rlike(wre(LEGAL + ORG))))

    # first names from P2P names: leading vs trailing counts per token
    p2p = named.where((F.col("cpty_type") == "P2P_CPTY") & F.col("alpha") & F.col("n_tok").between(2, 3) & ~F.col("biz_tok"))
    toks = F.split(F.col("nm_p"), " ")
    lead = F.when(F.col("has_comma") & F.col("ftc").isNotNull(), F.col("ftc")).otherwise(F.element_at(toks, 1))
    trail = F.when(F.col("has_comma"), F.element_at(toks, 1)).otherwise(F.element_at(toks, -1))
    fn_stats = (p2p.select(lead.alias("tok"), F.lit(1).alias("lead"), F.lit(0).alias("trail"))
                .unionByName(p2p.select(trail.alias("tok"), F.lit(0).alias("lead"), F.lit(1).alias("trail")))
                .groupBy("tok").agg(F.sum("lead").alias("lead"), F.sum("trail").alias("trail"))
                .where((F.length("tok") >= 2) & (F.col("lead") >= FN_MIN_IDS)
                       & (F.col("lead") >= FN_LEAD_RATIO * F.greatest(F.col("trail"), F.lit(1))))
                .where(~F.col("tok").isin(NOT_FIRST + LEGAL + ORG))
                .toPandas())
    FIRST_NAMES = set(fn_stats.tok)
    FN_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"tok": sorted(FIRST_NAMES)}), "tok string"))

    # learned business words: frequent in organisation names, rare in person names
    def tok_counts(df, col, label):
        return (df.select("cpty_id", F.explode(F.array_distinct(F.split(F.col(col), " "))).alias("tok"))
                  .where(F.col("tok").rlike(r"^[A-Z]{3,}$")).groupBy("tok").agg(F.count("*").alias(label)))
    biz_src = named.where(F.col("biz_tok") & ~F.coalesce(F.col("v1_ph"), F.lit(False))
                          & ~F.coalesce(F.col("v1_fi"), F.lit(False)) & ~F.col("nm").isin(PLACEHOLDERS))
    vocab = (tok_counts(biz_src, "nm_c", "biz_ids").join(tok_counts(p2p, "nm_p", "person_ids"), "tok", "left")
             .fillna(0, ["person_ids"])
             .where((F.col("biz_ids") >= VOCAB_MIN_IDS) & (F.col("biz_ids") >= VOCAB_RATIO * (F.col("person_ids") + 1)))
             .where(~F.col("tok").isin(STOP + LEGAL + PERSON_EXTRA))
             .toPandas())
    vocab = vocab[~vocab.tok.isin(FIRST_NAMES)]
    BIZ_VOCAB = set(vocab.tok)

    TOKV = F.broadcast(spark.createDataFrame(pd.DataFrame(
        [(t, t in FIRST_NAMES, t in BIZ_VOCAB) for t in sorted(FIRST_NAMES | BIZ_VOCAB)], columns=["tok", "is_fn", "is_bv"]),
        "tok string, is_fn boolean, is_bv boolean"))
    tok_flags = (named.select("cpty_id", F.posexplode(F.slice(F.split(F.col("nm_p"), " "), 1, 8)).alias("pos", "tok"))
                      .join(TOKV, "tok", "inner")
                      .groupBy("cpty_id")
                      .agg(F.max((F.col("is_fn") & (F.col("pos") == 0)).cast("int")).alias("fn_t1"),
                           F.max((F.col("is_fn") & (F.col("pos") == 1)).cast("int")).alias("fn_t2"),
                           F.max(F.col("is_fn").cast("int")).alias("fn_any"),
                           F.max(F.col("is_bv").cast("int")).alias("bv_any")))
    tok_flags.write.mode("overwrite").parquet(f"{TMP}/tok_flags")          # materialised: typing is not one huge stage

    # typer rules, in order
    fnc = FN_DF.select(F.col("tok").alias("ftc"), F.lit(True).alias("fnc"))
    alnum = F.length(F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", ""))
    digits = F.length(F.regexp_replace(F.col("nm"), r"[^0-9]", ""))
    wire_only = F.col("cats").rlike(r"^4[A-C](,4[A-C])*$")
    t = (named.join(fnc, "ftc", "left").join(spark.read.parquet(f"{TMP}/tok_flags"), "cpty_id", "left")
         .withColumn("fnc", F.coalesce("fnc", F.lit(False)))
         .fillna(0, ["fn_t1", "fn_t2", "fn_any", "bv_any"]))
    nc = F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", "")
    fc = F.regexp_replace(F.upper(F.coalesce(F.col("fi"), F.lit(""))), r"[^A-Z0-9]", "")
    name_is_fi = ((F.length(nc) >= 6) & (F.length(fc) >= 6)
                  & ((F.expr("left(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', ''), "
                             "length(regexp_replace(nm, '[^A-Z0-9]', '')))") == nc)
                     | (F.expr("left(regexp_replace(nm, '[^A-Z0-9]', ''), "
                               "length(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', '')))") == fc)))
    shape = F.col("alpha") & F.col("n_tok").between(2, 4) & ~F.col("nm_p").rlike(wre(["THE", "AT", "OF", "ST"]))
    can_person = F.col("amt") < PERSON_MAX_AMT
    c = (F.when(F.col("name").isNull() | (F.col("nm") == ""), F.array(F.lit("no_name"), F.lit("no_name")))
          .when(F.coalesce(F.col("v1_ph"), F.lit(False)) | F.col("nm").isin(PLACEHOLDERS) | F.col("nm").rlike(TEST_NAME_RE),
                F.array(F.lit("placeholder"), F.lit("placeholder")))
          .when((alnum > 0) & (digits >= NUM_SHARE * alnum), F.array(F.lit("numeric_name"), F.lit("numeric")))
          .when(F.coalesce(F.col("v1_fi"), F.lit(False)), F.array(F.lit("bank"), F.lit("bank_name")))
          .when(name_is_fi, F.array(F.lit("bank"), F.lit("name_is_fi")))
          .when(F.col("nm_c").rlike(wre(LEGAL)), F.array(F.lit("business"), F.lit("legal_suffix")))
          .when(F.col("nm_c").rlike(wre(ORG + ORG_BRANDS)), F.array(F.lit("business"), F.lit("org_word")))
          .when(F.col("bv_any") == 1, F.array(F.lit("business"), F.lit("learned_word")))
          .when(F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR", F.array(F.lit("business"), F.lit("ach_originator")))
          .when(F.col("cpty_type") == "MERCHANT", F.array(F.lit("business"), F.lit("merchant")))
          .when(can_person & F.col("has_comma") & shape & F.col("fnc"), F.array(F.lit("person"), F.lit("comma_name")))
          .when(can_person & shape & (F.col("fn_t1") == 1), F.array(F.lit("person"), F.lit("first_name")))
          .when(can_person & shape & (F.col("fn_t2") == 1), F.array(F.lit("person"), F.lit("last_first")))
          .when(can_person & F.col("alpha") & F.col("nm_p").contains(" AND ") & (F.col("n_tok") <= 8)
                & (F.col("fn_any") == 1), F.array(F.lit("person"), F.lit("joint_names")))
          .when(wire_only & (F.col("amt") >= WIRE_INST_MIN), F.array(F.lit("business"), F.lit("wire_institution")))
          .when(~can_person, F.array(F.lit("business"), F.lit("high_value")))
          .when((F.col("cpty_type") == "P2P_CPTY") & shape, F.array(F.lit("person"), F.lit("p2p_shape")))
          .otherwise(F.array(F.lit("unknown"), F.lit("unknown"))))
    # merge key: legal and channel words and reference numbers removed; digit-only names keep their digits
    k = F.regexp_replace(F.col("nm_c"), r"\bAT T\b", "ATT")
    k = F.regexp_replace(k, r"\b(" + "|".join(KEEP_NUM_AFTER) + r") (\d+)\b", "$1_$2")
    k = F.regexp_replace(k, r"\b([A-Z]{3,})(\d+)\b", "$1 $2")
    k = F.regexp_replace(k, r"([A-Z])(\d{4,})", "$1 $2")
    k = F.regexp_replace(k, r"(\d{3,})([A-Z])", "$1 $2")
    k = F.regexp_replace(k, r"\b(?=(?:[A-Z]*\d){4})[A-Z0-9]+\b", " ")
    k = F.regexp_replace(k, r"\b(?=[A-Z0-9]{5,}\b)(?=[A-Z]*\d[A-Z]*\d)[A-Z0-9]+\b", " ")
    k = F.regexp_replace(k, r"\b\d+\b", " ")
    k = F.regexp_replace(k, wre(LEGAL + CHANNEL + ["THE"]), " ")
    k = F.trim(F.regexp_replace(k, r"\s+", " "))
    key = F.regexp_replace(k, r"(\s+[A-Z0-9])+$", "")
    for _pat, _canon in KEY_ALIASES.items():
        key = F.when(key.rlike(_pat), F.lit(_canon)).otherwise(key)
    (t.withColumn("_c", c)
      .select("cpty_id", "cpty_id_src", "ent_l1", "name", "fi", "cpty_type", "cats", "amt", "n_txn",
              F.col("nm").alias("name_clean"), F.col("_c")[0].alias("name_class"), F.col("_c")[1].alias("type_reason"),
              F.when(F.col("_c")[0] == "business", key).alias("biz_key"),
              F.when(F.col("_c")[0] == "numeric_name", F.regexp_replace(F.col("nm"), r"[^0-9]", "")).alias("num_key"))
      .write.mode("overwrite").parquet(f"{TMP}/typed"))
    typed = spark.read.parquet(f"{TMP}/typed")

cls = typed.groupBy("name_class").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).toPandas()
cls = cls.sort_values("amt", ascending=False)
cls["share_ids"] = cls.ids / cls.ids.sum(); cls["share_dollars"] = cls.amt / cls.amt.sum(); cls["dollars_B"] = cls.amt / 1e9
n_l1 = typed.agg(F.countDistinct("ent_l1")).first()[0]
out(sec(f"§2 ent_l1 and typer v2 ({TIMES['§2 ent_l1 + typer']:.1f} min)",
        tbl(cls[["name_class", "ids", "share_ids", "dollars_B", "share_dollars"]], pct=("share_ids", "share_dollars"),
            dec=("dollars_B",))),
    para(f"{len(typed.columns)} columns typed for {cls.ids.sum():,} ids; {n_l1:,} ent_l1 units; "
         f"{len(FIRST_NAMES):,} first names and {len(BIZ_VOCAB):,} learned business words derived from the data."))

## §3 Entity map v2: `pkg_cpty_entity_map_v2`

- **`ent_l2`:** business ids merge on their cleaned name key. Distinctive keys (a word of ≥ 4 letters used by
  ≤ `DF_MAX` keys, or ≥ `CONC_MIN` ids per FI on average) merge across banks (`N:`, rule `business_name`); generic keys
  merge only within one bank (`F:`, rule `business_name_fi`); digit-only names merge on digits + bank (`D:`, rule
  `numeric_fi`). People, banks and placeholders stay at `ent_l1`.
- **`KEY_IGNORE_SPACES`:** keys that differ only by spaces form one merge key; it is distinctive when any of its
  spellings is distinctive, and its concentration counts all spellings. Its entity id is built from its most common
  spelling, so every key without spelling variants keeps the same `ent_l2` id as before.
- **`ent_l2_fi`:** the same merges, but always within one bank.
- **One row per id:** if an id ever has more than one row, the lowest `ent_l2` is kept and reported below.

In [ ]:
# ---- §3 Entity map v2 ------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

spark.conf.set("spark.sql.shuffle.partitions", str(2 * SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "32m")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
KNS = F.regexp_replace(F.col("biz_key"), " ", "") if KEY_IGNORE_SPACES else F.col("biz_key")
ALIAS_NS = sorted({(v.replace(" ", "") if KEY_IGNORE_SPACES else v) for v in KEY_ALIASES.values()})
with timed("§3 entity map"):
    blocked_words = sorted((set(LEGAL + ORG + GENERIC_KEYS + CHANNEL + NO_MERGE_KEYS) | FIRST_NAMES) - set(SHORT_KEYS_OK))
    BL_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"biz_key": blocked_words}), "biz_key string")
                         .withColumn("blk_word", F.lit(True)))
    has_key = F.col("biz_key").isNotNull() & (F.col("biz_key") != "")
    kb = typed.where(has_key).select("biz_key", "fi", "type_reason").withColumn("key_ns", KNS)

    # distinctiveness: document frequency of each word over the distinct business keys
    keys = kb.select("biz_key", "key_ns").distinct()
    kdf = (keys.select("biz_key", F.explode(F.array_distinct(F.split("biz_key", " "))).alias("tok"))
               .where(F.col("tok").rlike(r"^[A-Z]{4,}$")))
    tok_df = kdf.groupBy("tok").agg(F.count("*").alias("df"))
    common = set(LEGAL + ORG + CHANNEL + GENERIC_KEYS) | FIRST_NAMES
    key_rare = (kdf.join(tok_df, "tok").where((F.col("df") <= DF_MAX) & ~F.col("tok").isin(sorted(common)))
                   .select("biz_key").distinct())
    rare_ns = keys.join(key_rare, "biz_key").select("key_ns").distinct().withColumn("rare_ns", F.lit(True))
    fi_n = F.coalesce(F.upper(F.trim(F.col("fi"))), F.lit("(none)"))
    per_key = kb.groupBy("biz_key").agg(
        F.count("*").alias("k_ids"), F.countDistinct(fi_n).alias("k_fis"),
        F.avg(F.col("type_reason").isin("merchant", "ach_originator").cast("double")).alias("beh_share"))
    per_ns = kb.groupBy("key_ns").agg(F.count("*").alias("ns_ids"), F.countDistinct(fi_n).alias("ns_fis"))
    # the id of a merged key is the md5 of its most common spelling, so keys without spelling variants keep their ids
    w_rep = Window.partitionBy("key_ns").orderBy(F.desc("k_ids"), "biz_key")
    key_rep = (per_key.join(keys, "biz_key").withColumn("_r", F.row_number().over(w_rep)).filter("_r = 1")
                      .select("key_ns", F.col("biz_key").alias("key_rep")))
    key_distinct = (per_key.join(keys, "biz_key").join(per_ns, "key_ns").join(rare_ns, "key_ns", "left")
                           .join(key_rep, "key_ns")
                           .withColumn("distinct_key", F.coalesce(F.col("rare_ns"), F.lit(False))
                                                       | (F.col("ns_ids") >= CONC_MIN * F.col("ns_fis"))
                                                       | F.col("key_ns").isin(ALIAS_NS))
                           .withColumn("brand_ok", (F.col("k_ids") >= CONC_MIN * F.col("k_fis"))
                                                   & (F.col("beh_share") >= BRAND_BEH_SHARE) & (F.length("biz_key") >= 3))
                           .select("biz_key", "key_ns", "key_rep", "distinct_key", "brand_ok"))
    key_distinct.write.mode("overwrite").parquet(f"{TMP}/keys")

    # merge targets (business and numeric ids only; null keys stay out of the key join)
    tsel = [F.col("cpty_id").alias("ent_l1"), F.col("name_class").alias("t_class"),
            F.col("biz_key"), F.col("num_key").alias("t_num"), F.col("fi").alias("t_fi")]
    tgt_biz = (typed.where(has_key).select(*tsel).join(BL_DF, "biz_key", "left")
                    .join(spark.read.parquet(f"{TMP}/keys"), "biz_key", "left"))
    tgt_num = (typed.where(~has_key & F.col("num_key").isNotNull()).select(*tsel)
                    .withColumn("blk_word", F.lit(None).cast("boolean"))
                    .withColumn("key_ns", F.lit(None).cast("string"))
                    .withColumn("key_rep", F.lit(None).cast("string"))
                    .withColumn("distinct_key", F.lit(None).cast("boolean"))
                    .withColumn("brand_ok", F.lit(None).cast("boolean")))
    (tgt_biz.unionByName(tgt_num)
            .withColumn("distinct_key", F.coalesce(F.col("distinct_key"), F.lit(False)))
            .withColumn("brand_ok", F.coalesce(F.col("brand_ok"), F.lit(False)) & ~F.col("biz_key").isin(NO_MERGE_KEYS))
            .withColumn("t_key_ok", F.col("biz_key").isNotNull()
                                    & ((((F.length("biz_key") >= KEY_MIN_LEN) | F.col("biz_key").isin(SHORT_KEYS_OK))
                                        & F.col("blk_word").isNull())
                                       | F.col("brand_ok")))
            .drop("blk_word")
            .write.mode("overwrite").parquet(f"{TMP}/targets"))
    tgt = spark.read.parquet(f"{TMP}/targets")

    fi_s = F.coalesce(F.upper(F.trim(F.col("t_fi"))), F.lit(""))
    emap = (typed.select("cpty_id", "cpty_id_src", "ent_l1", "name_class", "type_reason", "name_clean")
                 .join(tgt, "ent_l1", "left")
                 .withColumn("ent_l0", F.col("cpty_id"))
                 .withColumn("ent_l2", F.when((F.col("t_class") == "business") & F.col("t_key_ok") & F.col("distinct_key"),
                                              F.concat(F.lit("N:"), F.md5("key_rep")))
                                        .when((F.col("t_class") == "business") & F.col("t_key_ok"),
                                              F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "key_rep", fi_s))))
                                        .when((F.col("t_class") == "numeric_name") & (F.length("t_num") >= 4),
                                              F.concat(F.lit("D:"), F.md5(F.concat_ws("|", "t_num", fi_s))))
                                        .otherwise(F.col("ent_l1")))
                 .withColumn("ent_l2_fi", F.when(F.col("ent_l2").startswith("N:"),
                                                 F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "key_rep", fi_s))))
                                           .otherwise(F.col("ent_l2")))
                 .withColumn("rule", F.when(F.col("ent_l2").startswith("N:"), "business_name")
                                      .when(F.col("ent_l2").startswith("F:"), "business_name_fi")
                                      .when(F.col("ent_l2").startswith("D:"), "numeric_fi")
                                      .when(F.col("ent_l1") != F.col("cpty_id"), "crosswalk_6c_5c")
                                      .otherwise("identity"))
                 .withColumn("key_blocked", F.coalesce((F.col("t_class") == "business") & ~F.col("t_key_ok"), F.lit(False)))
                 .select("cpty_id", "cpty_id_src", "ent_l0", "ent_l1", "ent_l2", "ent_l2_fi", "rule",
                         "name_class", "type_reason", "name_clean", F.col("biz_key").alias("name_key"), "key_blocked"))
    emap.write.mode("overwrite").parquet(f"{TMP}/map_all")
    m_all = spark.read.parquet(f"{TMP}/map_all")
    # one row per cpty_id: keep the lowest ent_l2 (deterministic) when an id has several rows
    w1 = Window.partitionBy("cpty_id").orderBy("ent_l2", "rule")
    (m_all.withColumn("_r", F.row_number().over(w1)).filter("_r = 1").drop("_r")
          .write.mode("overwrite").parquet(S["cpty_entity_map_v2"]))
    MAP = spark.read.parquet(S["cpty_entity_map_v2"])
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

n_dim = DIM.count()
chk = MAP.agg(F.count("*").alias("rows"), F.countDistinct("cpty_id").alias("ids"), F.countDistinct("ent_l1").alias("l1"),
              F.countDistinct("ent_l2").alias("l2"), F.countDistinct("ent_l2_fi").alias("l2fi")).first()
dups = (m_all.groupBy("cpty_id").agg(F.count("*").alias("rows"), F.collect_set("ent_l2").alias("ent_l2_values"))
             .where("rows > 1"))
n_dup = dups.count()
assert chk["rows"] == chk["ids"] == n_dim, f"map v2: {chk['rows']:,} rows, {chk['ids']:,} ids, dim {n_dim:,} ids"
levels = pd.DataFrame([{"level": "raw ids (ent_l0)", "units": chk["ids"]}, {"level": "ent_l1", "units": chk["l1"]},
                       {"level": "ent_l2", "units": chk["l2"]}, {"level": "ent_l2_fi", "units": chk["l2fi"]}])
levels["ids_per_unit"] = chk["ids"] / levels.units
rules = MAP.groupBy("rule").agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("units")).toPandas()
joined = (spark.read.parquet(f"{TMP}/keys").groupBy("key_ns")
               .agg(F.countDistinct("biz_key").alias("spellings"), F.concat_ws(" | ", F.sort_array(F.collect_set("biz_key"))).alias("keys"))
               .where("spellings > 1")
               .join(MAP.where(F.col("rule").isin("business_name", "business_name_fi"))
                        .withColumn("key_ns", F.regexp_replace("name_key", " ", "") if KEY_IGNORE_SPACES else F.col("name_key"))
                        .groupBy("key_ns").agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("units")), "key_ns")
               .orderBy(F.desc("ids")))
n_joined = joined.count()
top_joined = joined.limit(10).toPandas()
out(sec(f"§3 pkg_cpty_entity_map_v2 (staged, {TIMES['§3 entity map']:.1f} min)", tbl(levels, dec=("ids_per_unit",))),
    sec("Ids and units per rule", tbl(rules.sort_values("ids", ascending=False))),
    para(f"Ids with more than one map row before the reduction: {n_dup:,}" + (" (lowest ent_l2 kept)." if n_dup else ".")),
    sec(f"Keys joined by KEY_IGNORE_SPACES: {n_joined:,} (largest 10)", tbl(top_joined[["keys", "spellings", "ids", "units"]]))
        if KEY_IGNORE_SPACES and n_joined else para("No keys joined by KEY_IGNORE_SPACES."))

## §4 Canonical bank names: `pkg_fi_routing` and `pkg_fi_name_map`

- **Per routing number:** the canonical name is its dominant name (by dollars) in the latest month it appears;
  spellings are unified only within a routing number; routing numbers carrying the same set of names get the same
  canonical name.
- **Per raw name** (for ids without a routing prefix): the canonical name of the routing numbers it appears under, when
  one canonical name holds ≥ `NAME_MIN_SHARE` of its dollars; otherwise the raw name.
- Before both: `FI_FIXES` (exact names), `FI_FIX_RULES` (patterns), fintech partner names → the partner bank, division
  names → the parent bank, parent names cut short by the feed completed.

Downstream, the canonical bank of a counterparty id is `pkg_fi_routing` via its routing prefix, else `pkg_fi_name_map`
via its raw bank name, else the raw name. Same logic as `pkg_fi_canonical.ipynb`.

In [ ]:
# ---- §4 Canonical bank names -----------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def fi_key(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    k = re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()
    k = re.sub(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", k)
    for w in sorted(FI_STRIP, key=len, reverse=True):
        k = re.sub(rf"\b{w}\b", " ", k)
    k = re.sub(r"\s+", " ", k).strip()
    return k or None
_clean = lambda x: re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", str(x).upper())).strip()

with timed("§4 canonical banks"):
    cm = spark.table(T_CPTY_MONTH)
    # every raw FI name with its dollars (for FI_FIX_RULES targets)
    name_amt = (cm.where(F.col("fi").isNotNull()).groupBy("fi")
                  .agg(F.sum(F.col("amount").cast("double")).alias("amt")).toPandas())
    FIXES = dict(FI_FIXES)
    rule_rows = []
    for src_pat, tgt_pat in FI_FIX_RULES:
        cand = name_amt[name_amt.fi.map(lambda x: bool(re.search(tgt_pat, _clean(x))))].sort_values("amt", ascending=False)
        srcs = name_amt[name_amt.fi.map(lambda x: bool(re.search(src_pat, _clean(x))))]
        target = cand.fi.iloc[0] if len(cand) else None
        for s_name, s_amt in zip(srcs.fi, srcs.amt):
            if target is not None and s_name != target:
                FIXES[s_name] = target
            rule_rows.append({"rule": src_pat, "raw name matched": s_name, "dollars_B": s_amt / 1e9,
                              "now named": target if target is not None else "(no target name found: left as is)"})
        if not len(srcs):
            rule_rows.append({"rule": src_pat, "raw name matched": "(none)", "dollars_B": 0.0,
                              "now named": target if target is not None else "(no target name found)"})

    def fi_fix(s):
        # exact fixes and rule fixes, then fintech partner names → the partner bank, division names → the parent
        if s is None or (isinstance(s, float) and np.isnan(s)):
            return s
        s = FIXES.get(s, s)
        m = re.match(FI_PARTNER_RE, _clean(s)) or re.match(FI_DIVISION_RE, _clean(s))
        return m.group(1).strip() if m else s

    def complete_truncated(fixed, known):
        known = sorted(set(known))
        res = {}
        for f in set(fixed) - set(known):
            if f is None or (isinstance(f, float) and np.isnan(f)) or len(str(f)) < 6:
                continue
            hits = [k for k in known if k.startswith(f + "") and k != f]
            if len(hits) == 1:
                res[f] = hits[0]
        return res

    # routing × month × FI name
    rt_raw = (cm.where(F.col("fi").isNotNull())
                .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
                .where(F.col("routing") != "")
                .groupBy("routing", "month", "fi")
                .agg(F.sum(F.col("amount").cast("double")).alias("amt"), F.countDistinct("cpty_id").alias("ids"))
                .toPandas())
    rt_raw["fi_fixed"] = rt_raw.fi.map(fi_fix)
    dim_names = (DIM.where(F.col("fi").isNotNull()).groupBy("fi").agg(F.count("*").alias("ids")).toPandas())
    cm_names = name_amt[["fi"]]
    dim_names = dim_names.merge(cm_names, on="fi", how="outer").fillna({"ids": 0})
    dim_names["ids"] = dim_names.ids.astype("int64")
    dim_names["fi_fixed"] = dim_names.fi.map(fi_fix)
    _known = [str(x).upper() for x in dim_names.fi.dropna()]
    _div_tails = [fi_fix(x) for x in pd.concat([dim_names.fi, rt_raw.fi]).dropna().unique()
                  if re.match(FI_DIVISION_RE, _clean(x))]
    TRUNC_FIX = complete_truncated(_div_tails, _known)
    rt_raw["fi_fixed"] = rt_raw.fi_fixed.replace(TRUNC_FIX)
    dim_names["fi_fixed"] = dim_names.fi_fixed.replace(TRUNC_FIX)
    rt_raw["key"] = rt_raw.fi_fixed.map(fi_key)
    dim_names["key"] = dim_names.fi_fixed.map(fi_key)

    # canonical name per routing number
    rk = (rt_raw.dropna(subset=["key"]).groupby(["routing", "key", "fi_fixed"]).amt.sum().reset_index()
                .sort_values(["routing", "key", "amt"], ascending=[True, True, False]).drop_duplicates(["routing", "key"]))
    RK_SPELL = {(r, k): sp for r, k, sp in zip(rk.routing, rk.key, rk.fi_fixed)}
    last_m = rt_raw.groupby("routing").month.max().rename("last_month")
    first_m = rt_raw.groupby("routing").month.min().rename("first_month")
    latest = (rt_raw.merge(last_m, left_on=["routing", "month"], right_on=["routing", "last_month"])
                    .sort_values(["routing", "amt"], ascending=[True, False]).drop_duplicates("routing")).set_index("routing")
    kspan = rt_raw.groupby(["routing", "key"]).agg(k_first=("month", "min"), k_last=("month", "max"), k_amt=("amt", "sum"))
    rows = []
    for rt, g in rt_raw.groupby("routing"):
        keys_ = g.key.dropna().unique()
        raw = sorted(g.fi.unique())
        lk = latest.loc[rt, "key"]
        canon = RK_SPELL.get((rt, lk), latest.loc[rt, "fi_fixed"])
        if len(raw) == 1:
            ctype, switch = "single name", None
        elif len(keys_) == 1:
            ctype, switch = "spelling variant", None
        elif lk is None or lk not in kspan.loc[rt].index:
            ctype, switch = "concurrent names", None
        else:
            ks = kspan.loc[rt]
            new_first = ks.loc[lk, "k_first"]
            others_last = ks.drop(index=lk).k_last.max()
            ctype = "renamed / acquired" if others_last < new_first else "concurrent names"
            switch = new_first if ctype == "renamed / acquired" else None
        rows.append({"routing": rt, "fi_canonical": canon, "n_names": len(raw), "n_keys": len(keys_),
                     "change_type": ctype, "switch_month": switch, "names_seen": " | ".join(raw[:6]),
                     "first_month": first_m[rt], "last_month": last_m[rt],
                     "ids": int(g.groupby("month").ids.max().max()), "amt": float(g.amt.sum())})
    ROUT = pd.DataFrame(rows)
    ROUT["change_type"] = ROUT.change_type.replace({"renamed / acquired": "multiple names", "concurrent names": "multiple names"})
    kset = rt_raw.dropna(subset=["key"]).groupby("routing").key.apply(lambda s: "|".join(sorted(set(s))))
    ROUT["key_set"] = ROUT.routing.map(kset)
    multi = ROUT[ROUT.change_type == "multiple names"]
    for ks, grp in multi.groupby("key_set"):
        if len(grp) < 2:
            continue
        sub = rt_raw[rt_raw.routing.isin(grp.routing)]
        lm = sub.month.max()
        best = sub[sub.month == lm].groupby("key").amt.sum().idxmax()
        ROUT.loc[grp.index, "fi_canonical"] = sub[sub.key == best].groupby("fi_fixed").amt.sum().idxmax()
    ROUT["kept_raw"] = ROUT.routing.isin(FI_KEEP_RAW_ROUTINGS)
    ROUT.loc[ROUT.kept_raw, "fi_canonical"] = None

    # raw name → canonical name
    rt_c = rt_raw.merge(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]], on="routing")
    by = rt_c.groupby(["fi", "fi_canonical"]).agg(amt=("amt", "sum"), routings=("routing", "nunique")).reset_index()
    by["share"] = by.amt / by.groupby("fi").amt.transform("sum").clip(lower=1e-9)
    by = by.sort_values(["fi", "amt"], ascending=[True, False])
    top = by.drop_duplicates("fi")[["fi", "fi_canonical", "share"]].rename(columns={"fi_canonical": "top_canonical"})
    NAMEMAP = dim_names[["fi", "fi_fixed", "key", "ids"]].merge(top, on="fi", how="left")
    clear = NAMEMAP.share >= NAME_MIN_SHARE
    NAMEMAP["source"] = np.select([NAMEMAP.top_canonical.isna(), clear], ["no routing", "routing"], "ambiguous")
    NAMEMAP["fi_canonical"] = np.where(clear, NAMEMAP.top_canonical, NAMEMAP.fi_fixed)
    NAMEMAP["changed"] = NAMEMAP.fi != NAMEMAP.fi_canonical

    # stage both tables (explicit schema; NaN → NULL in text columns)
    def to_sdf(pdf):
        typ = {"object": "string", "bool": "boolean", "int64": "bigint", "int32": "int", "float64": "double"}
        pdf = pdf.copy()
        for c in pdf.columns:
            if pdf[c].dtype == object:
                pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
        schema = ", ".join(f"`{c}` {typ.get(str(pdf[c].dtype), 'string')}" for c in pdf.columns)
        return spark.createDataFrame(pdf, schema)
    rt_out = ROUT.drop(columns=["amt", "switch_month"]).assign(amount=ROUT.amt)
    nm_out = NAMEMAP[["fi", "fi_canonical", "source", "ids"]].rename(columns={"fi": "fi_raw"})
    to_sdf(rt_out).coalesce(1).write.mode("overwrite").parquet(S["fi_routing"])
    to_sdf(nm_out).coalesce(1).write.mode("overwrite").parquet(S["fi_name_map"])

assert ROUT.routing.is_unique and NAMEMAP.fi.is_unique, "bank tables must be one row per key"
ct = ROUT.groupby("change_type").agg(routing_numbers=("routing", "count"), amt=("amt", "sum")).reset_index()
ct["share_dollars"] = ct.amt / ct.amt.sum()
src = NAMEMAP.groupby("source").agg(raw_names=("fi", "count"), ids=("ids", "sum"), changed=("changed", "sum")).reset_index()
canon_of = dict(zip(NAMEMAP.fi, NAMEMAP.fi_canonical))
rr = pd.DataFrame(rule_rows)
if len(rr):
    rr["canonical (raw name)"] = rr["raw name matched"].map(canon_of)
    rr["canonical (target)"] = rr["now named"].map(canon_of)
out(sec(f"§4 pkg_fi_routing: {len(ROUT):,} routing numbers (staged, {TIMES['§4 canonical banks']:.1f} min)",
        tbl(ct[["change_type", "routing_numbers", "share_dollars"]], pct=("share_dollars",))),
    sec(f"pkg_fi_name_map: {len(NAMEMAP):,} raw names, {NAMEMAP.fi_canonical.nunique():,} canonical names", tbl(src)),
    sec("FI_FIX_RULES: raw names matched and the name they now take", tbl(rr, dec=("dollars_B",))) if len(rr) else "",
    para(f"FI_FIXES applied: {len(FI_FIXES)} exact, {len(FIXES) - len(FI_FIXES)} from rules. "
         f"Division names completed from truncation: {len(TRUNC_FIX)}."))

## §5 Customer dimension: `pkg_cust_dim`

One row per `mdm_id`: the latest `load_dt` from `neo4j_customer` (ties: the row with a `customer_pwr_id`), plus every
`mdm_id` seen as a customer in the edges but missing from `neo4j_customer`. Same logic as customer metrics v3.

In [ ]:
# ---- §5 Customer dimension -------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

CUST_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
             "customer_start_dt", "naics_cd", "naics_desc", "load_dt"]
with timed("§5 customer dimension"):
    EM_all = spark.table(T_EDGE_M)
    raw = (spark.table(CUST_TABLE).select(*[nz(c).alias(c) for c in CUST_COLS])
                .filter(F.col("mdm_id").isNotNull()).withColumn("party_type", F.upper("party_type")))
    w = Window.partitionBy("mdm_id").orderBy(F.col("load_dt").desc_nulls_last(), F.col("customer_pwr_id").isNotNull().desc(),
                                             F.col("customer_pwr_id").asc_nulls_last())
    latest = (raw.withColumn("n_load_versions", F.count("*").over(Window.partitionBy("mdm_id")))
                 .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
    nc = F.upper(F.col("naics_cd"))
    latest = (latest
        .withColumn("naics_status", F.when(nc.isNull(), "missing")
                                     .when(nc.isin(NAICS_PLACEHOLDERS) | ~nc.rlike(r"^[0-9]{2}"), "placeholder").otherwise("valid"))
        .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring("naics_cd", 1, 2)))
        .withColumn("is_tm", F.col("customer_pwr_id").isNotNull()))
    fam = latest.filter(F.col("master_pwr_id").isNotNull()).groupBy("master_pwr_id").agg(F.count("*").alias("family_size"))
    shared = (latest.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id")
                    .agg(F.count("*").alias("_n")).filter("_n > 1").select("customer_pwr_id", F.lit(True).alias("pwr_shared")))
    latest = (latest.join(fam, "master_pwr_id", "left").join(shared, "customer_pwr_id", "left")
                    .withColumn("in_neo4j_customer", F.lit(True)))
    seen = (EM_all.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), "month")
              .unionByName(EM_all.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), "month"))
              .groupBy("mdm_id").agg(F.min("month").alias("first_edge_month"), F.max("month").alias("last_edge_month")))
    cdim = (latest.join(seen, "mdm_id", "full")
             .withColumn("in_neo4j_customer", F.coalesce("in_neo4j_customer", F.lit(False)))
             .withColumn("is_tm", F.coalesce("is_tm", F.lit(False)))
             .withColumn("pwr_shared", F.coalesce("pwr_shared", F.lit(False)))
             .withColumn("naics_status", F.coalesce("naics_status", F.lit("missing")))
             .select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                     "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "pwr_shared", "family_size",
                     "customer_start_dt", "load_dt", "n_load_versions", "in_neo4j_customer",
                     "first_edge_month", "last_edge_month"))
    cdim.write.mode("overwrite").parquet(S["cust_dim"])
    CD = spark.read.parquet(S["cust_dim"])

q = CD.agg(F.count("*").alias("mdm_ids"), F.countDistinct("mdm_id").alias("distinct_ids"),
           F.sum(F.col("in_neo4j_customer").cast("long")).alias("in_neo4j_customer"),
           F.sum((~F.col("in_neo4j_customer")).cast("long")).alias("edges_only"),
           F.sum(F.col("first_edge_month").isNotNull().cast("long")).alias("seen_in_edges"),
           F.sum(F.col("is_tm").cast("long")).alias("tm_customers"),
           F.sum(F.col("pwr_shared").cast("long")).alias("pwr_shared_rows"),
           F.countDistinct("master_pwr_id").alias("master_pwr_ids"), F.max("load_dt").alias("latest_load_dt")).toPandas()
assert int(q.mdm_ids[0]) == int(q.distinct_ids[0]), "pkg_cust_dim must have one row per mdm_id"
out(sec(f"§5 pkg_cust_dim (staged, {TIMES['§5 customer dimension']:.1f} min)", tbl(q.drop(columns=["distinct_ids"]))))

## §6 Publish

Every staged table replaces the current one together: the current folder is set aside, the staged folder is moved into
place, and the table definition is re-created as an external table. Then the temporary folders are removed. Run the
printed statements in Impala.

In [ ]:
# ---- §6 Publish ------------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

with timed("§6 publish"):
    actions = []
    for k in OUT_KEYS:
        t, final, staged = T[k], P[k], S[k]
        assert exists(f"{staged}/_SUCCESS"), f"{k} was not staged"
        old_loc = None
        if table_exists(t):
            old_loc = table_location(t)
            kind = [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Type"][0].strip().upper()
            if kind == "EXTERNAL":
                spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('external.table.purge' = 'false')")
            spark.sql(f"DROP TABLE {t}")              # external: definition only; managed: its files too
        trash = f"{final}__old"
        delete(trash)
        if exists(final):
            rename(final, trash)
        rename(staged, final)
        cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(final).schema.fields)
        spark.sql(f"CREATE EXTERNAL TABLE {t} ({cols}) STORED AS PARQUET LOCATION '{final}' "
                  "TBLPROPERTIES ('external.table.purge' = 'false')")
        spark.catalog.refreshTable(t)
        delete(trash)
        n = spark.table(t).count()
        actions.append({"table": t, "rows": n, "GB": size_gb(final),
                        "note": "" if old_loc in (None, final) else f"previous location {old_loc} left as is"})
    delete(STAGE); delete(TMP)

pub = pd.DataFrame(actions)
times = pd.DataFrame([{"step": k, "minutes": v} for k, v in TIMES.items()])
out(sec("§6 Published", tbl(pub, dec=("GB",))),
    sec(f"Time: {times.minutes.sum():.1f} min", tbl(times, dec=("minutes",))),
    para("Impala: " + " ".join(f"INVALIDATE METADATA {T[k]};" for k in OUT_KEYS)))